# R2-MJ-14 — directionality checks on two variants where a trade-off is expected

Round-1 referee comment:

> I'm quite surprised that 93% of the pleiotropic lead variants showed fully concordant
> directionality across all of their associated diseases. (I assume that they have aligned the
> alleles and they are referring to alleles, rather than variants). I would consider it more
> plausible that a lot of the lead variant alleles that increased, for example, risk of
> autoimmunity, would decrease risk of infection. Is it really the case there are not other
> examples of balancing selection? Another example is PCSK9. The alleles at PCSK9 that increase a
> diagnosis of hypercholesterolemia should increase the risk of T2D. Is this not the case?

We are not defending full concordance: 92.5% is not a claim that antagonistic pleiotropy is rare.
Each check below takes **one variant where a trade-off is expected** and asks whether the method
detects it.

1. **PCSK9 `rs11591147` (R46L)** — does the allele that lowers hypercholesterolaemia risk raise type
   2 diabetes risk?
2. **TYK2 `rs34536443` (P1104A)** — does the allele that protects against autoimmune disease raise
   tuberculosis risk? Fallback if tuberculosis is not linked: **FUT2 `rs601338` (W143X)**,
   non-secretor allele, expected to protect against enteric infection and raise Crohn's disease risk.

No scan, no modelling.

## Alleles, not variants

The referee's parenthesis: every direction below names an **effect allele** — the alternative allele
of the variant identifier `chrom_pos_ref_alt`, which is what studies are harmonised to at ingestion.
Credible sets sharing a `variantId` therefore already share an effect allele, and nothing is
realigned here. Harmonisation itself is a Methods statement and is not re-verified.

The signed effect is

```
beta = rescaledStatistics.directionOfEffect * rescaledStatistics.absEstimatedBeta
```

`directionOfEffect` is `sign(originalBeta)` (`src/manuscript_methods/rescaled_beta.py`), so the sign
is the sign the study reported for that effect allele; the magnitude is the rescaled (log-odds)
effect. `originalBeta` is carried through every table so the raw reported value stays visible.

**Concordance, as in the paper** (Methods): the largest proportion of same-direction effects per
variant, over the credible sets of that variant that report a beta, range 0.5–1, and 1 for
non-pleiotropic variants.

In [1]:
from collections import defaultdict

import numpy as np
import pandas as pd
import pyarrow.compute as pc
import pyarrow.dataset as ds

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.max_rows", 200)

INTERMEDIATE = "../../../data/intermediate_files/"
RELEASE = "../../../data/25.06/"

# The three variants, with the window used to find them in the variant index.
VARIANTS = {
    "rs11591147": {"gene": "PCSK9", "protein_change": "R46L", "window": ("1", 54_900_000, 55_200_000)},
    "rs34536443": {"gene": "TYK2", "protein_change": "P1104A", "window": ("19", 10_200_000, 10_500_000)},
    "rs601338": {"gene": "FUT2", "protein_change": "W143X", "window": ("19", 48_600_000, 48_800_000)},
}

TYPE_2_DIABETES = "MONDO_0005148"
TUBERCULOSIS_ROOT = "MONDO_0018076"
INFECTIOUS_ROOT = "EFO_0005741"
HYPERCHOLESTEROLAEMIA = "HP_0003124"
CROHNS_DISEASE = "EFO_0000384"

## The corpus

`qualifying_credible_sets` is the disease credible-set table the pleiotropy analysis is built on
(`chapters/03-manuscript-figures/figure_3/python_scripts/variant_pleiotropy_analysis.ipynb`): one row
per qualifying credible set, carrying the lead `variantId`, the study's `diseaseIds`, the reported
`originalBeta` and the rescaled statistics.

In [2]:
cs_cols = [
    "studyId",
    "studyLocusId",
    "variantId",
    "variant",
    "diseaseIds",
    "originalBeta",
    "originalStandardError",
    "rescaledStatistics",
    "studyStatistics",
    "leadVariantConsequence",
    "majorLdPopulationMaf",
    "nCases",
    "nControls",
]
cs = ds.dataset(INTERMEDIATE + "qualifying_credible_sets", format="parquet").to_table(columns=cs_cols).to_pandas()

rescaled = pd.DataFrame(list(cs["rescaledStatistics"]))
study_stats = pd.DataFrame(list(cs["studyStatistics"]))

cs["beta"] = rescaled["directionOfEffect"].to_numpy() * rescaled["absEstimatedBeta"].to_numpy()
cs["se"] = rescaled["estimatedSE"].to_numpy()
cs["maf"] = cs["majorLdPopulationMaf"].apply(lambda m: None if m is None else m["value"])
cs["trait"] = study_stats["trait"].to_numpy()
cs["chromosome"] = cs["variant"].apply(lambda v: v["chromosome"])
cs["position"] = cs["variant"].apply(lambda v: v["start"])
cs["effectAllele"] = cs["variant"].apply(lambda v: v["alt"])
cs["otherAllele"] = cs["variant"].apply(lambda v: v["ref"])
cs = cs.drop(columns=["variant", "rescaledStatistics", "studyStatistics", "majorLdPopulationMaf"])

disease = pd.read_parquet(RELEASE + "output/disease")
DISEASE_NAME = dict(zip(disease["id"], disease["name"]))

cs_exploded = cs.explode("diseaseIds").rename(columns={"diseaseIds": "diseaseId"}).dropna(subset=["diseaseId"])

print(f"qualifying credible sets: {len(cs):,}")
print(f"unique lead variants:     {cs['variantId'].nunique():,}")

qualifying credible sets: 70,618
unique lead variants:     40,706


In [3]:
def paper_concordance(frame: pd.DataFrame) -> float:
    """Largest proportion of same-direction effects, over credible sets reporting a beta.

    The Methods definition, and the same aggregation as the figure-3 notebook: the denominator is
    the number of credible sets with a reported beta; a variant with no reported beta scores 1.
    """
    reported = frame.drop_duplicates("studyLocusId")
    reported = reported.loc[reported["originalBeta"].notna(), "beta"].dropna()
    if len(reported) == 0:
        return 1.0
    positive = float((reported > 0).mean())
    return max(positive, 1.0 - positive)


def closure(root: str) -> set[str]:
    """A class root plus every descendant of it in the 25.06 disease index."""
    row = disease.loc[disease["id"] == root]
    if row.empty:
        raise KeyError(f"{root} is not in the 25.06 disease index")
    descendants = row["descendants"].iloc[0]
    return {root} | set(descendants if descendants is not None else [])


def consequence_of(variant_id: str) -> str:
    """Most severe lead-variant consequence, as consequence type / gene / amino-acid change."""
    rows = cs.loc[cs["variantId"] == variant_id, "leadVariantConsequence"].dropna()
    if rows.empty:
        return "not annotated"
    severe = rows.iloc[0]["mostSevereConsequence"] or {}
    transcript = severe.get("transcriptConsequence") or {}
    return f"{severe.get('type')} / {transcript.get('approvedSymbol')} {transcript.get('aminoAcidChange')}"


def find_variant(rsid: str) -> pd.Series:
    """Resolve an rsID to a 25.06 variant record, searching its window in the variant index."""
    chrom, start, end = VARIANTS[rsid]["window"]
    hits = (
        ds.dataset(RELEASE + "output/variant", format="parquet")
        .to_table(
            columns=["variantId", "chromosome", "position", "referenceAllele", "alternateAllele", "rsIds"],
            filter=((pc.field("chromosome") == chrom) & (pc.field("position") > start) & (pc.field("position") < end)),
        )
        .to_pandas()
    )
    hits = hits[hits["rsIds"].apply(lambda ids: ids is not None and rsid in set(ids))]
    if len(hits) != 1:
        raise LookupError(f"{rsid} resolved to {len(hits)} variants")
    return hits.iloc[0]


def associations_of(variant_id: str, rsid: str) -> pd.DataFrame:
    """Every disease association of one lead variant, one row per credible set x disease."""
    rows = cs_exploded[cs_exploded["variantId"] == variant_id].copy()
    rows["rsId"] = rsid
    rows["gene"] = VARIANTS[rsid]["gene"]
    rows["diseaseName"] = rows["diseaseId"].map(DISEASE_NAME)
    rows["effectDirection"] = np.select(
        [rows["beta"] > 0, rows["beta"] < 0],
        ["increases risk", "decreases risk"],
        default="no beta reported",
    )
    return rows[
        [
            "rsId",
            "gene",
            "variantId",
            "effectAllele",
            "otherAllele",
            "studyId",
            "trait",
            "diseaseId",
            "diseaseName",
            "originalBeta",
            "originalStandardError",
            "beta",
            "se",
            "effectDirection",
            "nCases",
            "nControls",
            "studyLocusId",
        ]
    ].sort_values("beta")

### Colocalisation clusters

Needed only to answer "is the predicted trait linked to this variant's cluster at all". Clusters are
the Methods rule: a credible set, every credible set it colocalises with (coloc H4 ≥ 0.8 or eCAVIAR
CLPP ≥ 0.01), then every credible set sharing a lead variant with anything already in the cluster,
repeated to closure. Only chromosomes 1 and 19 are scanned — the three variants live there.

In [4]:
adjacency = defaultdict(set)
qualifying_loci = set(cs["studyLocusId"])

for chrom in sorted({VARIANTS[r]["window"][0] for r in VARIANTS}):
    coloc = (
        ds.dataset(RELEASE + "output/colocalisation_coloc", format="parquet")
        .to_table(
            columns=["leftStudyLocusId", "rightStudyLocusId"],
            filter=(pc.field("chromosome") == chrom) & (pc.field("h4") >= 0.8),
        )
        .to_pandas()
    )
    ecaviar = (
        ds.dataset(RELEASE + "output/colocalisation_ecaviar", format="parquet")
        .to_table(
            columns=["leftStudyLocusId", "rightStudyLocusId"],
            filter=(pc.field("chromosome") == chrom) & (pc.field("clpp") >= 0.01),
        )
        .to_pandas()
    )
    edges = pd.concat([coloc, ecaviar], ignore_index=True)
    edges = edges[edges["leftStudyLocusId"].isin(qualifying_loci) & edges["rightStudyLocusId"].isin(qualifying_loci)]
    for left, right in edges.itertuples(index=False):
        adjacency[left].add(right)
        adjacency[right].add(left)
    print(f"chromosome {chrom}: {len(edges):,} colocalisation edges between qualifying credible sets")

loci_of_variant = defaultdict(set)
for variant_id, locus_id in cs[["variantId", "studyLocusId"]].itertuples(index=False):
    loci_of_variant[variant_id].add(locus_id)
variant_of_locus = dict(cs[["studyLocusId", "variantId"]].itertuples(index=False))


def cluster_of(variant_id: str) -> pd.DataFrame:
    """The credible sets of the colocalisation cluster containing `variant_id`."""
    seen: set[str] = set()
    stack = list(loci_of_variant[variant_id])
    while stack:
        locus = stack.pop()
        if locus in seen:
            continue
        seen.add(locus)
        stack.extend(adjacency[locus] - seen)
        stack.extend(loci_of_variant[variant_of_locus[locus]] - seen)
    return cs_exploded[cs_exploded["studyLocusId"].isin(seen)]

chromosome 1: 82,149 colocalisation edges between qualifying credible sets


chromosome 19: 72,733 colocalisation edges between qualifying credible sets


# Check 1 — PCSK9 `rs11591147` (R46L)

**Hypothesis**: the allele that lowers hypercholesterolaemia risk raises type 2 diabetes risk — the
two associations discordant on the same allele.

In [5]:
pcsk9 = find_variant("rs11591147")
pcsk9_id = pcsk9["variantId"]
pcsk9_assoc = associations_of(pcsk9_id, "rs11591147")

print(f"rs11591147 -> {pcsk9_id}  (ref {pcsk9['referenceAllele']}, alt {pcsk9['alternateAllele']})")
print(f"lead variant of qualifying credible sets: {pcsk9_assoc['studyLocusId'].nunique()}")
print(f"unique diseases: {pcsk9_assoc['diseaseId'].nunique()}")
print(f"effect allele: {pcsk9_assoc['effectAllele'].iloc[0]}   other allele: {pcsk9_assoc['otherAllele'].iloc[0]}")
print(f"lead variant consequence: {consequence_of(pcsk9_id)}")
print(f"MAF (major LD population): {cs.loc[cs['variantId'] == pcsk9_id, 'maf'].max():.4f}")
print(f"\nconcordance (paper formula): {paper_concordance(pcsk9_assoc):.3f}")

rs11591147 -> 1_55039974_G_T  (ref G, alt T)
lead variant of qualifying credible sets: 60
unique diseases: 21
effect allele: T   other allele: G
lead variant consequence: in-gene-effect / PCSK9 R46L
MAF (major LD population): 0.0392

concordance (paper formula): 0.964


## The lipid side of the hypothesis

In [6]:
lipid_terms = pcsk9_assoc["diseaseName"].str.contains(
    "cholesterol|lipid|lipoprotein|lipidemia|lipidaemia|glycerid", case=False, na=False
)
lipid = pcsk9_assoc[lipid_terms]
print(
    f"hypercholesterolaemia ({HYPERCHOLESTEROLAEMIA}) credible sets: "
    f"{(pcsk9_assoc['diseaseId'] == HYPERCHOLESTEROLAEMIA).sum()}"
)
print(
    lipid[["studyId", "trait", "diseaseId", "diseaseName", "originalBeta", "beta", "effectDirection"]].to_string(
        index=False
    )
)

summary_by_disease = (
    pcsk9_assoc.groupby(["diseaseId", "diseaseName"])
    .agg(
        n_credible_sets=("studyLocusId", "nunique"),
        n_increases=("beta", lambda s: int((s > 0).sum())),
        n_decreases=("beta", lambda s: int((s < 0).sum())),
        min_beta=("beta", "min"),
        max_beta=("beta", "max"),
    )
    .reset_index()
    .sort_values("n_credible_sets", ascending=False)
)
print(f"\nall {len(summary_by_disease)} diseases on the T allele of rs11591147:")
print(summary_by_disease.to_string(index=False))

hypercholesterolaemia (HP_0003124) credible sets: 7
                   studyId                                                          trait       diseaseId                            diseaseName  originalBeta      beta effectDirection
              GCST90104007    Familial combined hyperlipidemia defined by Mexico criteria   MONDO_0001336                familial hyperlipidemia     -0.010664 -0.505327  decreases risk
              GCST90104006 Familial combined hyperlipidemia defined by Consensus criteria   MONDO_0001336                familial hyperlipidemia     -0.026754 -0.497398  decreases risk
     FINNGEN_R12_E4_FH_IHD     Familial hypercholesterolemia, with ischemic heart disease     EFO_0004911          familial hypercholesterolemia     -0.562713 -0.464599  decreases risk
         FINNGEN_R12_E4_FH                                  Familial hypercholesterolemia     EFO_0004911          familial hypercholesterolemia     -0.471852 -0.440361  decreases risk
FINNGEN_R12_E4_HYPERLIP

## The type 2 diabetes side of the hypothesis

In [7]:
t2d_corpus = cs_exploded[cs_exploded["diseaseId"] == TYPE_2_DIABETES]
pcsk9_cluster = cluster_of(pcsk9_id)

print(f"{TYPE_2_DIABETES} ({DISEASE_NAME[TYPE_2_DIABETES]}):")
print(
    f"  qualifying credible sets corpus-wide: {len(t2d_corpus):,} "
    f"on {t2d_corpus['variantId'].nunique():,} lead variants"
)
print(f"  on rs11591147:                        {(pcsk9_assoc['diseaseId'] == TYPE_2_DIABETES).sum()}")
print(f"  in rs11591147's colocalisation cluster: {(pcsk9_cluster['diseaseId'] == TYPE_2_DIABETES).sum()}")
print(
    f"\ncluster: {pcsk9_cluster['studyLocusId'].nunique()} credible sets, "
    f"{pcsk9_cluster['variantId'].nunique()} lead variants, "
    f"{pcsk9_cluster['diseaseId'].nunique()} diseases"
)
print(f"cluster diseases: {sorted(pcsk9_cluster['diseaseId'].map(DISEASE_NAME).dropna().unique())}")

MONDO_0005148 (type 2 diabetes mellitus):
  qualifying credible sets corpus-wide: 4,757 on 3,367 lead variants
  on rs11591147:                        0
  in rs11591147's colocalisation cluster: 0

cluster: 62 credible sets, 2 lead variants, 23 diseases
cluster diseases: ['Abdominal Aortic Aneurysm', 'Disorder of lipid metabolism', 'Hypercholesterolemia', 'Hyperlipidemia', 'Myocardial Ischemia', 'acute myocardial infarction', 'angina pectoris', 'atherosclerosis', 'cardiovascular disease', 'coronary artery bypass', 'coronary artery disease', 'coronary atherosclerosis', 'familial hypercholesterolemia', 'familial hyperlipidemia', 'familial lipoprotein lipase deficiency', 'heart disease', 'hyperlipidemia', 'intermediate coronary syndrome', 'metabolic disease', 'myocardial infarction', 'percutaneous transluminal coronary angioplasty', 'response to statin', 'vascular disease']


**Result.** The hypothesis cannot be tested at this locus, and the premise does not hold in our data
either. Type 2 diabetes has thousands of credible sets corpus-wide and none on `rs11591147`, none in
its cluster — that is coverage, not direction, and no proxy is substituted for it. Separately, the T
allele *lowers* hypercholesterolaemia rather than raising it, as expected for a loss-of-function
allele, so the referee's premise ("alleles that increase a diagnosis of hypercholesterolemia") has no
carrier here. The variant is nonetheless not fully concordant: its concordance is below 1 because two
credible sets run the opposite way to the other 54.

# Check 2 — TYK2 `rs34536443` (P1104A)

**Hypothesis**: the allele that protects against autoimmune disease raises tuberculosis risk —
discordant on the same allele.

In [8]:
tyk2 = find_variant("rs34536443")
tyk2_id = tyk2["variantId"]
tyk2_assoc = associations_of(tyk2_id, "rs34536443")

print(f"rs34536443 -> {tyk2_id}  (ref {tyk2['referenceAllele']}, alt {tyk2['alternateAllele']})")
print(f"lead variant of qualifying credible sets: {tyk2_assoc['studyLocusId'].nunique()}")
print(f"unique diseases: {tyk2_assoc['diseaseId'].nunique()}")
print(f"effect allele: {tyk2_assoc['effectAllele'].iloc[0]}   other allele: {tyk2_assoc['otherAllele'].iloc[0]}")
print(f"lead variant consequence: {consequence_of(tyk2_id)}")
print(f"MAF (major LD population): {cs.loc[cs['variantId'] == tyk2_id, 'maf'].max():.4f}")
print(f"\nconcordance (paper formula): {paper_concordance(tyk2_assoc):.3f}")

rs34536443 -> 19_10352442_G_C  (ref G, alt C)
lead variant of qualifying credible sets: 54
unique diseases: 19
effect allele: C   other allele: G
lead variant consequence: in-gene-effect / TYK2 P1104A
MAF (major LD population): 0.0445

concordance (paper formula): 0.974


## Every autoimmune association, on the C allele

In [9]:
autoimmune_of_interest = {
    "EFO_0000685": "rheumatoid arthritis",
    "EFO_0009459": "ACPA-positive rheumatoid arthritis",
    "MONDO_0007915": "systemic lupus erythematosus",
    "MONDO_0005147": "type 1 diabetes mellitus",
    "EFO_0000676": "psoriasis",
    "EFO_1001494": "psoriasis vulgaris",
    "EFO_0003898": "ankylosing spondylitis",
    "EFO_0000384": "Crohn's disease",
    "EFO_0005140": "autoimmune disease",
    "EFO_0003779": "Hashimoto's thyroiditis",
    "EFO_0006812": "autoimmune thyroid disease",
    "MONDO_0019338": "sarcoidosis",
    "EFO_0004705": "hypothyroidism",
    "EFO_0005856": "arthritis",
}

tyk2_summary = (
    tyk2_assoc.groupby(["diseaseId", "diseaseName"])
    .agg(
        n_credible_sets=("studyLocusId", "nunique"),
        n_increases=("beta", lambda s: int((s > 0).sum())),
        n_decreases=("beta", lambda s: int((s < 0).sum())),
        n_no_beta=("beta", lambda s: int(s.isna().sum())),
        min_beta=("beta", "min"),
        max_beta=("beta", "max"),
    )
    .reset_index()
)
tyk2_summary["hypothesised_autoimmune"] = tyk2_summary["diseaseId"].isin(autoimmune_of_interest)
print(f"all {len(tyk2_summary)} diseases on the C allele of rs34536443:")
print(tyk2_summary.sort_values(["hypothesised_autoimmune", "n_credible_sets"], ascending=False).to_string(index=False))

missing = [f"{i} ({n})" for i, n in autoimmune_of_interest.items() if i not in set(tyk2_assoc["diseaseId"])]
print(f"\nhypothesised autoimmune terms with no credible set on this variant: {missing}")

print("\nindividual autoimmune credible sets, effect allele C:")
print(
    tyk2_assoc[tyk2_assoc["diseaseId"].isin(autoimmune_of_interest)][
        ["studyId", "trait", "diseaseId", "diseaseName", "originalBeta", "beta", "effectDirection"]
    ].to_string(index=False)
)

all 19 diseases on the C allele of rs34536443:
    diseaseId                        diseaseName  n_credible_sets  n_increases  n_decreases  n_no_beta  min_beta  max_beta  hypothesised_autoimmune
  EFO_0000676                          psoriasis               11            0            8          3 -0.458174 -0.272383                     True
  EFO_0000685               rheumatoid arthritis               11            0            7          4 -0.353893 -0.175628                     True
  EFO_0004705                     hypothyroidism                5            0            5          0 -0.141201 -0.069973                     True
  EFO_1001494                 psoriasis vulgaris                4            0            1          3 -0.345011 -0.345011                     True
MONDO_0019338                        sarcoidosis                3            0            3          0 -0.318428 -0.282920                     True
  EFO_0005140                 autoimmune disease                2

## Tuberculosis, and what else the C allele does to infection

In [10]:
tuberculosis = closure(TUBERCULOSIS_ROOT)
infectious = closure(INFECTIOUS_ROOT)
tb_corpus = cs_exploded[cs_exploded["diseaseId"].isin(tuberculosis)]
tyk2_cluster = cluster_of(tyk2_id)

print(f"tuberculosis ({TUBERCULOSIS_ROOT} + {len(tuberculosis) - 1} descendants):")
print(
    f"  qualifying credible sets corpus-wide: {len(tb_corpus)} "
    f"on {tb_corpus['variantId'].nunique()} lead variants, "
    f"{tb_corpus['diseaseId'].nunique()} distinct term(s), studies "
    f"{sorted(tb_corpus['studyId'].unique())}"
)
print(f"  on rs34536443:                        {tyk2_assoc['diseaseId'].isin(tuberculosis).sum()}")
print(f"  in rs34536443's colocalisation cluster: {tyk2_cluster['diseaseId'].isin(tuberculosis).sum()}")
print(
    f"\ncluster: {tyk2_cluster['studyLocusId'].nunique()} credible sets, "
    f"{tyk2_cluster['variantId'].nunique()} lead variants, "
    f"{tyk2_cluster['diseaseId'].nunique()} diseases"
)

print("\ninfection-related associations that the C allele does have:")
infection_like = tyk2_assoc[
    tyk2_assoc["diseaseId"].isin(infectious)
    | tyk2_assoc["diseaseName"].str.contains("tonsillitis|infect|pneumon|abscess", case=False, na=False)
]
print(
    infection_like[
        ["studyId", "trait", "diseaseId", "diseaseName", "originalBeta", "beta", "effectDirection"]
    ].to_string(index=False)
)
print(
    f"\nnote: tonsillitis (MONDO_0001039) is not a descendant of {INFECTIOUS_ROOT} in the 25.06 "
    "ontology — it sits under disorder of pharynx / upper respiratory tract disorder — so it is "
    "reported here on clinical grounds, named explicitly, not as an ontology-derived infection class."
)

tuberculosis (MONDO_0018076 + 37 descendants):
  qualifying credible sets corpus-wide: 5 on 5 lead variants, 1 distinct term(s), studies ['GCST000764', 'GCST001398', 'GCST002810', 'GCST004923', 'GCST90275067']
  on rs34536443:                        0
  in rs34536443's colocalisation cluster: 0

cluster: 136 credible sets, 19 lead variants, 28 diseases

infection-related associations that the C allele does have:
                    studyId                                                      trait     diseaseId diseaseName  originalBeta     beta  effectDirection
FINNGEN_R12_J10_TONSILLITIS                          Other and unspecified tonsillitis MONDO_0001039 tonsillitis      0.189472 0.190543   increases risk
               GCST90104030 COVID-19 (critical illness vs population or mild symptoms) MONDO_0100096    COVID-19           NaN      NaN no beta reported
               GCST90270934                  COVID-19 (critical illness vs population) MONDO_0100096    COVID-19           Na

**Result.** Tuberculosis is not linked: it has 5 qualifying credible sets in the whole corpus, on one
term and five different lead variants, none of them `rs34536443`, and none in its cluster. The
hypothesis as stated is therefore untestable here — coverage again, and the fallback variant is run
below as instructed.

The trade-off itself is still visible on this variant, on a different infection. Every autoimmune
association of the C allele is protective, and the one infection-coded association it carries,
tonsillitis, runs the other way. That single opposing credible set is exactly why its concordance is
below 1 — the method detects the trade-off where an association exists to detect it.

# Fallback — FUT2 `rs601338` (W143X)

Run because tuberculosis is not linked to TYK2. **Hypothesis**: the non-secretor allele protects
against enteric infection and raises Crohn's disease risk — discordant on the same allele.

In [11]:
fut2 = find_variant("rs601338")
fut2_id = fut2["variantId"]
fut2_assoc = associations_of(fut2_id, "rs601338")

print(f"rs601338 -> {fut2_id}  (ref {fut2['referenceAllele']}, alt {fut2['alternateAllele']})")
print(f"lead variant of qualifying credible sets: {fut2_assoc['studyLocusId'].nunique()}")
print(f"unique diseases: {fut2_assoc['diseaseId'].nunique()}")
print(f"effect allele: {fut2_assoc['effectAllele'].iloc[0]}   other allele: {fut2_assoc['otherAllele'].iloc[0]}")
print(f"lead variant consequence: {consequence_of(fut2_id)}")
print(f"MAF (major LD population): {cs.loc[cs['variantId'] == fut2_id, 'maf'].max():.4f}")
print(f"\nconcordance (paper formula): {paper_concordance(fut2_assoc):.3f}")

print("\nevery association of the A allele:")
print(
    fut2_assoc[["studyId", "trait", "diseaseId", "diseaseName", "originalBeta", "beta", "effectDirection"]].to_string(
        index=False
    )
)

rs601338 -> 19_48703417_G_A  (ref G, alt A)
lead variant of qualifying credible sets: 14
unique diseases: 12
effect allele: A   other allele: G
lead variant consequence: in-gene-effect / FUT2 W154*
MAF (major LD population): 0.4960

concordance (paper formula): 0.846

every association of the A allele:
        studyId                                                                         trait     diseaseId                              diseaseName  originalBeta      beta  effectDirection
   GCST90475774                                          Megaloblastic anemia (PheCode 281.1) MONDO_0001700                     megaloblastic anemia     -0.017879 -0.146554   decreases risk
   GCST90480951                                         Other deficiency anemia (PheCode 281) MONDO_0001639                        deficiency anemia     -0.017397 -0.119896   decreases risk
   GCST90079962                                   ICD10 I10: Essential (primary) hypertension MONDO_0001134                   

In [12]:
fut2_cluster = cluster_of(fut2_id)
fut2_infection = fut2_cluster[fut2_cluster["diseaseId"].isin(infectious)].copy()
fut2_infection["diseaseName"] = fut2_infection["diseaseId"].map(DISEASE_NAME)

print(
    f"Crohn's disease ({CROHNS_DISEASE}) on the A allele: "
    f"{(fut2_assoc['diseaseId'] == CROHNS_DISEASE).sum()} credible set(s), "
    f"beta {fut2_assoc.loc[fut2_assoc['diseaseId'] == CROHNS_DISEASE, 'beta'].tolist()}"
)
print(f"infectious-disease descendants on the A allele itself: {fut2_assoc['diseaseId'].isin(infectious).sum()}")
print(
    f"\ncluster: {fut2_cluster['studyLocusId'].nunique()} credible sets, "
    f"{fut2_cluster['variantId'].nunique()} lead variants, "
    f"{fut2_cluster['diseaseId'].nunique()} diseases"
)
print(f"infectious-disease descendants in the cluster, carried by other lead variants:")
print(
    fut2_infection[
        ["variantId", "effectAllele", "studyId", "trait", "diseaseId", "diseaseName", "originalBeta", "beta"]
    ]
    .sort_values(["diseaseName", "variantId"])
    .to_string(index=False)
)
print(
    "\nthese are other variants' alleles: their signs are stated with respect to their own effect "
    "alleles and cannot be carried over to rs601338's A allele without LD phase, which is not "
    "available here. They are reported as cluster coverage, not as a direction for rs601338."
)

Crohn's disease (EFO_0000384) on the A allele: 1 credible set(s), beta [0.10273134255189262]
infectious-disease descendants on the A allele itself: 0

cluster: 102 credible sets, 42 lead variants, 58 diseases
infectious-disease descendants in the cluster, carried by other lead variants:
      variantId effectAllele                                       studyId                                                                              trait     diseaseId                   diseaseName  originalBeta      beta
19_48697960_C_T            T                                  GCST90104030                         COVID-19 (critical illness vs population or mild symptoms) MONDO_0100096                      COVID-19      0.095310  0.129261
19_48700572_C_T            T                                  GCST90454507                                        COVID-19 (hospitalized covid vs population) MONDO_0100096                      COVID-19     -0.065700 -0.144356
19_48702851_C_G            G      

**Result.** The Crohn's side of the hypothesis holds on the named allele: the A (non-secretor) allele
raises Crohn's disease risk. The enteric-infection side is not testable on this variant — `rs601338`
carries no infectious-disease association of its own. Its cluster does contain intestinal infectious
disease, dysentery and COVID-19 credible sets, but on other lead variants, so those directions belong
to those alleles and are not transferable. Coverage again, stated rather than proxied.

# Summary table and outputs

In [13]:
checks = []
for rsid, variant_id, assoc, hypothesis, predicted, predicted_present, verdict in [
    (
        "rs11591147",
        pcsk9_id,
        pcsk9_assoc,
        "allele lowering hypercholesterolaemia raises type 2 diabetes risk",
        f"{TYPE_2_DIABETES} (type 2 diabetes mellitus)",
        bool((pcsk9_assoc["diseaseId"] == TYPE_2_DIABETES).any()),
        "untestable — predicted trait not linked to the variant or its cluster",
    ),
    (
        "rs34536443",
        tyk2_id,
        tyk2_assoc,
        "allele protecting against autoimmune disease raises tuberculosis risk",
        f"{TUBERCULOSIS_ROOT} (tuberculosis) + descendants",
        bool(tyk2_assoc["diseaseId"].isin(tuberculosis).any()),
        "untestable for tuberculosis; trade-off detected against tonsillitis on the same allele",
    ),
    (
        "rs601338",
        fut2_id,
        fut2_assoc,
        "non-secretor allele protects against enteric infection and raises Crohn's disease risk",
        f"{INFECTIOUS_ROOT} descendants (enteric infection)",
        bool(fut2_assoc["diseaseId"].isin(infectious).any()),
        "Crohn's side confirmed on the A allele; infection side not linked to the variant",
    ),
]:
    determined = assoc.drop_duplicates("studyLocusId")
    determined = determined[determined["originalBeta"].notna()]
    checks.append(
        {
            "rsId": rsid,
            "gene": VARIANTS[rsid]["gene"],
            "proteinChange": VARIANTS[rsid]["protein_change"],
            "variantId": variant_id,
            "effectAllele": assoc["effectAllele"].iloc[0],
            "otherAllele": assoc["otherAllele"].iloc[0],
            "maf_major_ld_population": round(float(cs.loc[cs["variantId"] == variant_id, "maf"].max()), 4),
            "n_credible_sets": int(assoc["studyLocusId"].nunique()),
            "n_diseases": int(assoc["diseaseId"].nunique()),
            "n_credible_sets_increasing_risk": int((determined["beta"] > 0).sum()),
            "n_credible_sets_decreasing_risk": int((determined["beta"] < 0).sum()),
            "concordance_paper_formula": round(paper_concordance(assoc), 3),
            "hypothesis": hypothesis,
            "predicted_trait": predicted,
            "predicted_trait_on_variant": predicted_present,
            "verdict": verdict,
        }
    )

checks = pd.DataFrame(checks)
checks.to_csv(INTERMEDIATE + "directionality_variant_checks-r1.csv", index=False)
print(checks.drop(columns=["hypothesis", "predicted_trait"]).to_string(index=False))

      rsId  gene proteinChange       variantId effectAllele otherAllele  maf_major_ld_population  n_credible_sets  n_diseases  n_credible_sets_increasing_risk  n_credible_sets_decreasing_risk  concordance_paper_formula  predicted_trait_on_variant                                                                                verdict
rs11591147 PCSK9          R46L  1_55039974_G_T            T           G                   0.0392               60          21                                2                               54                      0.964                       False                  untestable — predicted trait not linked to the variant or its cluster
rs34536443  TYK2        P1104A 19_10352442_G_C            C           G                   0.0445               54          19                                1                               38                      0.974                       False untestable for tuberculosis; trade-off detected against tonsillitis on the same allel

In [14]:
all_assoc = pd.concat([pcsk9_assoc, tyk2_assoc, fut2_assoc], ignore_index=True)
all_assoc.to_csv(INTERMEDIATE + "directionality_variant_associations-r1.csv", index=False)

cluster_rows = []
for rsid, variant_id, cluster, predicted_terms, predicted_label in [
    ("rs11591147", pcsk9_id, pcsk9_cluster, {TYPE_2_DIABETES}, "type 2 diabetes"),
    ("rs34536443", tyk2_id, tyk2_cluster, tuberculosis, "tuberculosis + descendants"),
    ("rs601338", fut2_id, fut2_cluster, infectious, "infectious disease descendants"),
]:
    hits = cluster[cluster["diseaseId"].isin(predicted_terms)]
    cluster_rows.append(
        {
            "rsId": rsid,
            "variantId": variant_id,
            "cluster_credible_sets": int(cluster["studyLocusId"].nunique()),
            "cluster_lead_variants": int(cluster["variantId"].nunique()),
            "cluster_diseases": int(cluster["diseaseId"].nunique()),
            "predicted_trait": predicted_label,
            "predicted_trait_credible_sets_in_cluster": int(hits["studyLocusId"].nunique()),
            "predicted_trait_lead_variants_in_cluster": int(hits["variantId"].nunique()),
            "carried_by_this_variant": bool((hits["variantId"] == variant_id).any()),
        }
    )

cluster_coverage = pd.DataFrame(cluster_rows)
cluster_coverage.to_csv(INTERMEDIATE + "directionality_cluster_coverage-r1.csv", index=False)
print(cluster_coverage.to_string(index=False))

      rsId       variantId  cluster_credible_sets  cluster_lead_variants  cluster_diseases                predicted_trait  predicted_trait_credible_sets_in_cluster  predicted_trait_lead_variants_in_cluster  carried_by_this_variant
rs11591147  1_55039974_G_T                     62                      2                23                type 2 diabetes                                         0                                         0                    False
rs34536443 19_10352442_G_C                    136                     19                28     tuberculosis + descendants                                         0                                         0                    False
  rs601338 19_48703417_G_A                    102                     42                58 infectious disease descendants                                        13                                         7                    False


# What the checks say

**`rs11591147`, PCSK9 R46L, effect allele T** (MAF 0.039 in its major LD population; lead-variant
consequence annotated `PCSK9 R46L`). A lead variant of 60 qualifying credible sets across 21
diseases. The T allele **lowers** hypercholesterolaemia (`HP_0003124`, 7 credible sets, β −0.17 to
−0.30), familial hypercholesterolaemia (−0.38 to −0.46), familial hyperlipidaemia (−0.37 to −0.51)
and coronary artery disease (−0.13 to −0.31) — the loss-of-function direction, and the opposite sign
to the premise in the comment, which assumes an allele that *increases* a hypercholesterolaemia
diagnosis. **Type 2 diabetes is not linked**: 4,757 credible sets corpus-wide on 3,367 lead variants,
none on this variant, none anywhere in its cluster (62 credible sets, 2 lead variants, 23 diseases).
The prediction is untestable here for coverage reasons and no proxy was substituted. Concordance is
**0.964**, not 1: two credible sets of the triglyceride-coded PheCode term (`MONDO_0009387`,
"Hyperglyceridemia", PheCode 272.12) run positive against 54 negative.

**`rs34536443`, TYK2 P1104A, effect allele C** (MAF 0.045, so well clear of the rare-variant
filters; consequence annotated `TYK2 P1104A`). A lead variant of 54 credible sets across 19 diseases.
The C allele is protective in **every** autoimmune association it carries: psoriasis (β −0.27 to
−0.46 across 8 credible sets), rheumatoid arthritis (−0.18 to −0.35 across 7), ACPA-positive
rheumatoid arthritis (−0.33, −0.37), systemic lupus erythematosus (−0.40), type 1 diabetes (−0.28,
−0.31), sarcoidosis (−0.28 to −0.32), psoriasis vulgaris (−0.35), hypothyroidism (−0.07 to −0.14),
Hashimoto's thyroiditis (−0.12), Crohn's disease (−0.26), arthritis (−0.15), autoimmune disease
(−0.16, −0.19). Ankylosing spondylitis has no credible set on this variant.

**Tuberculosis is not linked**: the whole corpus holds 5 tuberculosis credible sets, one term
(`MONDO_0018076`), five other lead variants, none in this variant's cluster of 136 credible sets. The
hypothesis as stated is untestable, so the fallback below was run. The trade-off is nonetheless
detectable on this allele against a different infection: the only association of the C allele that
runs positive is **tonsillitis, β = +0.19** — the same allele that protects against autoimmunity
raises risk of an infectious condition — and that single credible set is exactly why the concordance
is **0.974** rather than 1. Two further infection-relevant terms carry no usable direction: three
COVID-19 credible sets and two type 2 diabetes credible sets on this variant report no beta.
Tonsillitis is reported here on clinical grounds and named explicitly: `MONDO_0001039` is not a
descendant of `EFO_0005741` in the 25.06 ontology.

**`rs601338`, FUT2, effect allele A (non-secretor)** (MAF 0.50; the 25.06 consequence annotation
labels it `FUT2 W154*`, the same stop-gain the comment calls W143X under a different transcript
numbering). A lead variant of 14 credible sets across 12 diseases. **The Crohn's half of the
hypothesis holds on the named allele**: β = **+0.10**, increases risk — alongside type 1 diabetes
(+0.11, +0.13), duodenal ulcer (+0.15), gallstones (+0.08), cholelithiasis (+0.06),
hypercholesterolaemia (+0.05) and essential hypertension (+0.04); two anaemia terms
(megaloblastic −0.15, deficiency −0.12) run the other way, giving concordance **0.846**. **The
enteric-infection half is not testable on this variant**: it carries no infectious-disease
association of its own. Its cluster does contain 13 such credible sets — intestinal infectious
disease, dysentery, viral (intestinal) disease, COVID-19, peritonsillar abscess, tracheitis — but on
7 other lead variants, whose signs refer to their own effect alleles and cannot be carried across to
the A allele without LD phase, which is not available here.

**Taken together.** In all three cases the predicted counter-trait was absent from the variant
itself, so the specific predictions could not be evaluated; that is coverage, and it is reported as
coverage rather than papered over with a proxy. Where an opposing association does exist, the
pipeline registers it with no special handling: PCSK9 R46L and TYK2 P1104A both come out **below**
full concordance, each driven by the one association that runs against the variant's dominant
direction, and FUT2 sits at 0.846. The 92.5% headline reflects which trait pairs have been measured
in the corpus, not a claim that antagonistic pleiotropy is rare.